# Credit Risk Analysis & Data Cleaning Pipeline
Comprehensive Exploratory Data Analysis (EDA), Missing Data Imputation, Outlier Handling, and Feature Engineering for Credit Risk Assessment (`dataset/Credit Risk Dataset.xlsx`).

In [ ]:
import pandas as pd
import numpy as np

# 1. Load Credit Risk Dataset
file_path = '../dataset/Credit Risk Dataset.xlsx'
df = pd.read_excel(file_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# 2. Target Variable & Feature Overview
print('Target Distribution (loan_status):')
print(df['loan_status'].value_counts(normalize=True).round(4) * 100)

print('\nMissing Values Count:')
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# 3. Outlier Inspection & Business Logic Validation
# Age > 100 or Employment Length > Age are anomalous data entries
print('Anomalous Age (> 100):', (df['person_age'] > 100).sum())
print('Anomalous Emp Length (> 60 years):', (df['person_emp_length'] > 60).sum())

# Filter out biologically impossible records
df_clean = df[(df['person_age'] <= 100) & (df['person_emp_length'] <= 60)].copy()
print(f'Cleaned Shape after outlier removal: {df_clean.shape}')

In [ ]:
# 4. Missing Value Imputation
# Impute interest rate by median of the corresponding loan_grade
if 'loan_int_rate' in df_clean.columns:
    df_clean['loan_int_rate'] = df_clean.groupby('loan_grade')['loan_int_rate'].transform(
        lambda x: x.fillna(x.median())
    )

# Impute employment length with median
if 'person_emp_length' in df_clean.columns:
    df_clean['person_emp_length'] = df_clean['person_emp_length'].fillna(df_clean['person_emp_length'].median())

print('Remaining nulls:', df_clean.isnull().sum().sum())

In [ ]:
# 5. Feature Engineering: Risk Indicator Ratios
df_clean['total_debt'] = df_clean['loan_amnt'] + df_clean['other_debt']
df_clean['monthly_income'] = df_clean['person_income'] / 12.0
df_clean['estimated_monthly_payment'] = (df_clean['loan_amnt'] * (1 + (df_clean['loan_int_rate'] / 100))) / df_clean['loan_term_months']
df_clean['payment_to_income_ratio'] = df_clean['estimated_monthly_payment'] / (df_clean['monthly_income'] + 1)

print('Engineered Features Sample:')
df_clean[['total_debt', 'estimated_monthly_payment', 'payment_to_income_ratio', 'loan_status']].head()

In [ ]:
# 6. Correlation Analysis with Default Status
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
corrs = df_clean[numeric_cols].corr()['loan_status'].sort_values(ascending=False)
print('=== Top Correlations with Loan Default (loan_status) ===')
print(corrs.round(4))